In [119]:
import torch
import torch.optim as optim
import torch.nn as nn
import optuna
import numpy as np
from torch.utils.data import DataLoader
from torch.optim import Adam
from sklearn.metrics import f1_score, classification_report
import matplotlib.pyplot as plt
import kagglehub
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [120]:
torch.cuda.empty_cache()

# Set up the device to be used for training (CUDA if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Download latest version
path = kagglehub.dataset_download("subhaditya/fer2013plus")

print("Path to dataset files:", path)

Using device: cuda
Path to dataset files: C:\Users\margi\.cache\kagglehub\datasets\subhaditya\fer2013plus\versions\1


In [135]:
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter(log_dir="runs/fer2013plus/4conv_3connected_weights_sqrt_plus3epochs")

# Download latest version
path = kagglehub.dataset_download("subhaditya/fer2013plus")

print("Path to dataset files:", path)
# Set up data transformations for training and testing
data_transforms = {
    "train": transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.RandomCrop(48, padding=4),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ]),
    "test": transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
}

# Paths to dataset
train_dir = path + "/fer2013plus/fer2013/train"
test_dir = path + "/fer2013plus/fer2013/test"

# Load datasets using ImageFolder
train_dataset = datasets.ImageFolder(train_dir, transform=data_transforms["train"])
test_dataset = datasets.ImageFolder(test_dir, transform=data_transforms["test"])

print("Train and Test Datasets Loaded!")

Path to dataset files: C:\Users\margi\.cache\kagglehub\datasets\subhaditya\fer2013plus\versions\1
Train and Test Datasets Loaded!


In [122]:
class_counts = np.bincount(train_dataset.targets)
print("Class names:", train_dataset.classes)
print("Class Counts:", class_counts)

Class names: ['anger', 'contempt', 'disgust', 'fear', 'happiness', 'neutral', 'sadness', 'surprise']
Class Counts: [ 2466   165   191   652  7528 10308  3514  3562]


In [123]:



# Function to calculate class weights
def calculate_class_weights(train_dataset):
    class_counts = np.bincount(train_dataset.targets)
    class_weights = 1. / np.sqrt(class_counts)
    return torch.tensor(class_weights, dtype=torch.float32)


In [124]:
from torch.utils.data import DataLoader, WeightedRandomSampler
import numpy as np
import torch


# Function to create dataloaders with weighted sampling
def create_dataloader(batch_size):
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Calculate class weights to handle imbalanced dataset
    class_weights = calculate_class_weights(train_dataset)

    # Move class weights to the same device as the model
    class_weights = class_weights.to(device)

    # Create sample weights for training
    sample_weights = [class_weights[label] for label in train_dataset.targets]

    # Create WeightedRandomSampler for training
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

    # Recreate the train loader with the sampler
    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler)

    return train_loader, test_loader, class_weights


In [125]:
import torch
import torch.optim as optim
import torch.nn.functional as F


# Function to train the model
def train_model(model, train_loader, optimizer, criterion, device, epochs=10, enable_logging=True):
    model.train()

    train_accuracy = []
    train_f1 = []
    train_losses = []

    for epoch in range(epochs):
        correct = 0
        total = 0
        all_preds = []
        all_labels = []
        running_loss = 0.0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Backward pass
            loss.backward()
            optimizer.step()

            # Update running loss
            running_loss += loss.item()

            # Get predictions
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

            # Store predictions and true labels for F1 calculation
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

        # Calculate metrics for the epoch
        epoch_accuracy = 100 * correct / total
        epoch_f1 = f1_score(all_labels, all_preds, average='weighted')
        epoch_loss = running_loss / len(train_loader)

        train_accuracy.append(epoch_accuracy)
        train_f1.append(epoch_f1)
        train_losses.append(epoch_loss)


        if enable_logging:
            # Log to TensorBoard
            writer.add_scalar('Accuracy/train', epoch_accuracy, epoch)
            writer.add_scalar('F1/train', epoch_f1, epoch)
            writer.add_scalar('Loss/train', epoch_loss, epoch)

        # Print epoch metrics
        print(f"Epoch {epoch + 1}/{epochs}, Train Accuracy: {epoch_accuracy:.2f}%, Train F1 Score: {epoch_f1:.4f}, Loss: {epoch_loss:.4f}")

    return train_accuracy, train_f1



In [126]:
def evaluate_model(model, test_loader, device, class_names):
    model.eval()

    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    # Initialize a dictionary to store class-wise accuracy using class names
    per_class_accuracy = {class_name: {"correct": 0, "total": 0} for class_name in class_names}

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            # Forward pass
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)

            # Calculate overall accuracy
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

            # Store predictions and true labels for F1 calculation
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            # Calculate per-class accuracy
            for label, prediction in zip(labels, predicted):
                class_name = class_names[label.item()]
                per_class_accuracy[class_name]["total"] += 1
                if label == prediction:
                    per_class_accuracy[class_name]["correct"] += 1

    # Calculate accuracy and F1 score for the entire dataset
    accuracy = 100 * correct / total
    f1 = f1_score(all_labels, all_preds, average='weighted')

    # Calculate per-class accuracy using class names
    class_accuracies = {
        class_name: (per_class_accuracy[class_name]["correct"] / per_class_accuracy[class_name]["total"]) * 100
        if per_class_accuracy[class_name]["total"] > 0 else 0
        for class_name in class_names}

    # Log test metrics to TensorBoard
    writer.add_scalar('Accuracy/test', accuracy, 0)
    writer.add_scalar('F1/test', f1, 0)

    # Log per-class accuracies
    for class_name, accuracy in class_accuracies.items():
        writer.add_scalar(f'Accuracy/test/class_{class_name}', accuracy, 0)

    print(f"Test Accuracy: {accuracy:.2f}%, Test F1 Score: {f1:.4f}")
    for class_name, accuracy in class_accuracies.items():
        print(f"Class {class_name} Accuracy: {accuracy:.2f}%")

    return class_accuracies  # Return per-class accuracies as a dictionary


In [127]:
class CNN(nn.Module):
    def __init__(self, dropout_rate=0.5):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1)
        
        # Adding a new convolutional layer with output channels 256
        self.conv4 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, stride=1, padding=1)
        
        # Fully connected layers
        self.fc1 = nn.Linear(in_features=256 * 6 * 6, out_features=256)
        self.fc2 = nn.Linear(in_features=256, out_features=128)
        self.fc3 = nn.Linear(in_features=128, out_features=8)

        # Dropout layers
        self.dropout1 = nn.Dropout(dropout_rate)
        self.dropout2 = nn.Dropout(dropout_rate)
        self.dropout3 = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        
        # Adding the new convolutional layer
        x = F.relu(self.conv4(x))  # Output shape after conv4: (batch_size, 256, 6, 6)
        
        x = x.view(-1, 256 * 6 * 6)  # Flatten the tensor
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)  # Apply dropout after first fully connected layer
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)  # Apply dropout after second fully connected layer
        x = self.fc3(x)
        x = self.dropout3(x)  # Apply dropout after output layer
        return x


In [128]:
import optuna
from optuna.samplers import TPESampler


# Define the objective function for hyperparameter optimization
def objective(trial):
    # Define the hyperparameters to be tuned
    batch_size = trial.suggest_int('batch_size', 16, 128, step=16)
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-3, log=True)
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.5)
    epochs = trial.suggest_int('epochs', 3, 10)

    # Create DataLoader
    train_loader, test_loader, class_weights_tensor = create_dataloader(batch_size)

    # Create and move model to device
    model = CNN(dropout_rate).to(device)
    
    optimizer = Adam(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

    # Train the model
    train_model(model, train_loader, optimizer, criterion, device, epochs=epochs, enable_logging=False)

    # Evaluate the model and get per-class accuracy
    per_class_accuracy = evaluate_model(model, test_loader, device,train_dataset.classes)

    # Return the average accuracy across classes as the objective to be maximized
    return np.mean(list(per_class_accuracy.values()))  # Now it will work as per_class_accuracy is a dictionary


# Define a study to optimize the objective function
study = optuna.create_study(direction='maximize', sampler=TPESampler())
study.optimize(objective, n_trials=10)  # You can increase n_trials for more exploration

# Print the best hyperparameters found
print("Best hyperparameters: ", study.best_params)




[I 2024-12-07 16:41:49,926] A new study created in memory with name: no-name-9d146d2b-34a5-4735-9409-249df2dcd893


Epoch 1/10, Train Accuracy: 18.36%, Train F1 Score: 0.1806, Loss: 2.0688
Epoch 2/10, Train Accuracy: 22.10%, Train F1 Score: 0.2254, Loss: 2.0111
Epoch 3/10, Train Accuracy: 21.43%, Train F1 Score: 0.2254, Loss: 1.9601
Epoch 4/10, Train Accuracy: 24.29%, Train F1 Score: 0.2555, Loss: 1.9077
Epoch 5/10, Train Accuracy: 27.48%, Train F1 Score: 0.2869, Loss: 1.8611
Epoch 6/10, Train Accuracy: 29.07%, Train F1 Score: 0.3036, Loss: 1.8238
Epoch 7/10, Train Accuracy: 31.14%, Train F1 Score: 0.3262, Loss: 1.7782
Epoch 8/10, Train Accuracy: 32.59%, Train F1 Score: 0.3429, Loss: 1.7366
Epoch 9/10, Train Accuracy: 33.81%, Train F1 Score: 0.3540, Loss: 1.6989
Epoch 10/10, Train Accuracy: 35.10%, Train F1 Score: 0.3699, Loss: 1.6722


[I 2024-12-07 16:46:13,207] Trial 0 finished with value: 47.81856074288017 and parameters: {'batch_size': 32, 'learning_rate': 9.215767845559171e-05, 'dropout_rate': 0.33604140046874964, 'epochs': 10}. Best is trial 0 with value: 47.81856074288017.


Test Accuracy: 71.44%, Test F1 Score: 0.4999
Class anger Accuracy: 44.57%
Class contempt Accuracy: 60.78%
Class disgust Accuracy: 43.86%
Class fear Accuracy: 37.13%
Class happiness Accuracy: 61.52%
Class neutral Accuracy: 33.81%
Class sadness Accuracy: 29.44%
Class surprise Accuracy: 71.44%
Epoch 1/9, Train Accuracy: 12.41%, Train F1 Score: 0.1220, Loss: 2.0791
Epoch 2/9, Train Accuracy: 16.26%, Train F1 Score: 0.1591, Loss: 2.0714
Epoch 3/9, Train Accuracy: 17.01%, Train F1 Score: 0.1724, Loss: 2.0649
Epoch 4/9, Train Accuracy: 17.41%, Train F1 Score: 0.1839, Loss: 2.0518
Epoch 5/9, Train Accuracy: 18.35%, Train F1 Score: 0.1939, Loss: 2.0328
Epoch 6/9, Train Accuracy: 18.81%, Train F1 Score: 0.2006, Loss: 2.0174
Epoch 7/9, Train Accuracy: 18.97%, Train F1 Score: 0.2017, Loss: 2.0082
Epoch 8/9, Train Accuracy: 19.82%, Train F1 Score: 0.2105, Loss: 1.9955
Epoch 9/9, Train Accuracy: 19.77%, Train F1 Score: 0.2095, Loss: 1.9856


[I 2024-12-07 16:50:04,032] Trial 1 finished with value: 29.19411413778055 and parameters: {'batch_size': 128, 'learning_rate': 4.157966077559092e-05, 'dropout_rate': 0.4947909086844849, 'epochs': 9}. Best is trial 0 with value: 47.81856074288017.


...

Test Accuracy: 60.89%, Test F1 Score: 0.2854
Class anger Accuracy: 36.18%
Class contempt Accuracy: 11.76%
Class disgust Accuracy: 7.02%
Class fear Accuracy: 11.38%
Class happiness Accuracy: 15.60%
Class neutral Accuracy: 32.15%
Class sadness Accuracy: 7.13%
Class surprise Accuracy: 60.89%
Epoch 1/8, Train Accuracy: 17.12%, Train F1 Score: 0.1826, Loss: 2.0420
Epoch 2/8, Train Accuracy: 22.76%, Train F1 Score: 0.2379, Loss: 1.9252
Epoch 3/8, Train Accuracy: 27.41%, Train F1 Score: 0.2869, Loss: 1.8348
Epoch 4/8, Train Accuracy: 32.05%, Train F1 Score: 0.3336, Loss: 1.7351
Epoch 5/8, Train Accuracy: 35.22%, Train F1 Score: 0.3675, Loss: 1.6421
Epoch 6/8, Train Accuracy: 38.96%, Train F1 Score: 0.4050, Loss: 1.5698
Epoch 7/8, Train Accuracy: 41.72%, Train F1 Score: 0.4326, Loss: 1.4930
Epoch 8/8, Train Accuracy: 42.88%, Train F1 Score: 0.4444, Loss: 1.4404


[I 2024-12-07 17:11:51,790] Trial 8 finished with value: 47.865994884834414 and parameters: {'batch_size': 64, 'learning_rate': 0.00021665943555735937, 'dropout_rate': 0.21657344277669868, 'epochs': 8}. Best is trial 4 with value: 50.10950820468502.


Test Accuracy: 71.67%, Test F1 Score: 0.5018
Class anger Accuracy: 52.17%
Class contempt Accuracy: 37.25%
Class disgust Accuracy: 63.16%
Class fear Accuracy: 24.55%
Class happiness Accuracy: 53.53%
Class neutral Accuracy: 28.84%
Class sadness Accuracy: 51.75%
Class surprise Accuracy: 71.67%
Epoch 1/9, Train Accuracy: 20.69%, Train F1 Score: 0.1957, Loss: 2.0519
Epoch 2/9, Train Accuracy: 24.15%, Train F1 Score: 0.2451, Loss: 1.9708
Epoch 3/9, Train Accuracy: 26.22%, Train F1 Score: 0.2718, Loss: 1.9119
Epoch 4/9, Train Accuracy: 28.14%, Train F1 Score: 0.2920, Loss: 1.8495
Epoch 5/9, Train Accuracy: 29.93%, Train F1 Score: 0.3097, Loss: 1.8131
Epoch 6/9, Train Accuracy: 32.36%, Train F1 Score: 0.3345, Loss: 1.7576
Epoch 7/9, Train Accuracy: 34.17%, Train F1 Score: 0.3533, Loss: 1.7186
Epoch 8/9, Train Accuracy: 35.92%, Train F1 Score: 0.3719, Loss: 1.6732
Epoch 9/9, Train Accuracy: 37.71%, Train F1 Score: 0.3915, Loss: 1.6237


[I 2024-12-07 17:16:15,236] Trial 9 finished with value: 47.08778362920023 and parameters: {'batch_size': 16, 'learning_rate': 5.749811170268058e-05, 'dropout_rate': 0.23861128999240835, 'epochs': 9}. Best is trial 4 with value: 50.10950820468502.


Test Accuracy: 69.89%, Test F1 Score: 0.5177
Class anger Accuracy: 42.08%
Class contempt Accuracy: 49.02%
Class disgust Accuracy: 42.11%
Class fear Accuracy: 43.71%
Class happiness Accuracy: 64.26%
Class neutral Accuracy: 39.47%
Class sadness Accuracy: 26.17%
Class surprise Accuracy: 69.89%
Best hyperparameters:  {'batch_size': 32, 'learning_rate': 0.00021160451138760525, 'dropout_rate': 0.3662026826804355, 'epochs': 8}


In [136]:
# best_params_3x3_sqrt = {'batch_size': 32, 'learning_rate': 0.00019969325624696616, 'dropout_rate': 0.22158981707531503,
#                'epochs': 7}

# Step 1: Get the best hyperparameters from the optimization

best_params = study.best_params

batch_size = best_params[('batch_s'
                          'ize')]
learning_rate = best_params['learning_rate']
dropout_rate = best_params['dropout_rate']
epochs = best_params['epochs'] + 3

# Step 2: Create DataLoader with the best batch size
train_loader, test_loader, class_weights_tensor = create_dataloader(batch_size)

# Step 3: Create and initialize the model with the best dropout rate
model = CNN(dropout_rate).to(device)

# Step 4: Set up the optimizer and criterion (loss function)
optimizer = Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# Step 5: Train the model with the best hyperparameters
train_accuracy, train_f1 = train_model(model, train_loader, optimizer, criterion, device, epochs=epochs)

# Step 6: Evaluate the model's performance
test_accuracy = evaluate_model(model, test_loader, device, train_dataset.classes)

writer.close()

Epoch 1/11, Train Accuracy: 17.05%, Train F1 Score: 0.1789, Loss: 2.0709
Epoch 2/11, Train Accuracy: 20.95%, Train F1 Score: 0.2204, Loss: 2.0009
Epoch 3/11, Train Accuracy: 24.40%, Train F1 Score: 0.2586, Loss: 1.9041
Epoch 4/11, Train Accuracy: 28.92%, Train F1 Score: 0.3060, Loss: 1.8185
Epoch 5/11, Train Accuracy: 32.12%, Train F1 Score: 0.3401, Loss: 1.7309
Epoch 6/11, Train Accuracy: 34.51%, Train F1 Score: 0.3656, Loss: 1.6580
Epoch 7/11, Train Accuracy: 37.38%, Train F1 Score: 0.3943, Loss: 1.6066
Epoch 8/11, Train Accuracy: 37.61%, Train F1 Score: 0.3961, Loss: 1.5644
Epoch 9/11, Train Accuracy: 39.17%, Train F1 Score: 0.4100, Loss: 1.5233
Epoch 10/11, Train Accuracy: 41.27%, Train F1 Score: 0.4292, Loss: 1.4785
Epoch 11/11, Train Accuracy: 41.96%, Train F1 Score: 0.4361, Loss: 1.4415
Test Accuracy: 66.11%, Test F1 Score: 0.5502
Class anger Accuracy: 55.75%
Class contempt Accuracy: 50.98%
Class disgust Accuracy: 56.14%
Class fear Accuracy: 65.27%
Class happiness Accuracy: 66.9

In [137]:
writer.close()
torch.save(model.state_dict(), 'cnn_strat_hyper.pth')